# QORC — supervised attack classification on a fixed photonic reservoir

**Scope.** This notebook covers the **supervised head only**. The novelty /
zero-day head is handled separately by another member of the team. Both heads
are designed to sit on top of the *same* fixed quantum stack:

```
X (122 feat) --> scaler --> PCA(n_modes) --> tanh phase encode
                                                 |
                              precircuit -> phase dials -> reservoir     [FIXED]
                                                 |
                                          readout: C(m,p) probs
                                                 |
                        +------------------------+------------------------+
                        |                                                 |
                 NOVELTY HEAD                                   SUPERVISED HEAD
              (normal rows only)                              (attack labels)
              -- handled elsewhere --                          -- THIS NOTEBOOK --
```

The pre-circuit, encoding and reservoir are **identical** to
`qorc_zeroday_analysis.ipynb`. Nothing about the quantum stage is retrained;
only the classical layer after the readout changes, and the two heads are
trained completely separately.

---

## What this notebook varies

Instead of sweeping random seeds, we sweep **reservoir capacity** — the
(photons, modes) configuration that sets the readout dimension:

| config | photons | modes | readout dim = C(m,p) |
|---|---|---|---|
| `3p12m` | 3 | 12 | 220 |
| `3p20m` | 3 | 20 | 1140 |
| `3p24m` | 3 | 24 | 2024 |
| `4p24m` | 4 | 24 | 10626 |

## Two label sets, run for every config

1. **`families6`** — 5 named attack families + `other_attack`
   (`neptune`, `satan`, `ipsweep`, `portsweep`, `smurf`, `other_attack`)
   plus `normal`. Every class has >2,600 rows, so per-class F1 is stable.
2. **`all23`** — every label in the dataset as its own class (23 classes,
   including a 2-row `spy` class). Fine-grained, but the tail is
   statistically meaningless and we report it as such.

Both are evaluated against a **classical baseline** on the same PCA dials, so
every quantum number has a like-for-like control.

## 1. Imports, configuration & run directory

In [ ]:
import os, sys, math, json, time, hashlib, platform, warnings, traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score)

import torch
import perceval as pcvl
import merlin as ML
try:
    MeasurementStrategy = ML.MeasurementStrategy
    ComputationSpace    = ML.ComputationSpace
except AttributeError:
    from merlin.measurement.strategies import MeasurementStrategy, ComputationSpace

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

# ---------------------------------------------------------------- settings
SEED       = 42
TEST_SIZE  = 0.30
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reservoir configs to sweep: (tag, n_photons, n_modes)
CONFIGS = [
    ("3p12m", 3, 12),
    ("3p20m", 3, 20),
    ("3p24m", 3, 24),
    ("4p12m", 4, 12),
    ("4p20m", 4, 20),
    ("4p24m", 4, 24),
]

# Label schemes to run for every config
LABEL_SCHEMES = ["families6", "all23"]

# Named families for the `families6` scheme; everything else -> other_attack
NAMED_FAMILIES = ["neptune", "satan", "ipsweep", "portsweep", "smurf"]

# Head training (torch GPU softmax regression)
HEAD_MAX_ITER = 200
HEAD_L2       = 1e-4

# Feature-matrix guard: refuse to build anything above this many GB
MAX_FEATURE_GB = 6.0

np.random.seed(SEED)
torch.manual_seed(SEED)

# ---------------------------------------------------------------- run dir
RUN_ROOT = Path("qorc_sup_runs")
RUN_ROOT.mkdir(exist_ok=True)
RUN_ID   = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR  = RUN_ROOT / RUN_ID
RUN_DIR.mkdir(exist_ok=True)
(RUN_DIR / "artifacts").mkdir(exist_ok=True)

LOG_PATH = RUN_DIR / "run.log"

def log(msg, level="INFO"):
    """Timestamped log to stdout AND to disk (flushed immediately).

    Everything important goes through here so an overnight run that dies
    still leaves a complete, readable trace on disk.
    """
    line = f"{datetime.now().strftime('%H:%M:%S')} [{level:5s}] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a", encoding="utf-8") as fh:
        fh.write(line + "\n")
        fh.flush()
        os.fsync(fh.fileno())

log(f"run id      : {RUN_ID}")
log(f"run dir     : {RUN_DIR.resolve()}")
log(f"device      : {DEVICE}"
    + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
log(f"python      : {sys.version.split()[0]} | torch {torch.__version__}"
    f" | perceval {pcvl.__version__}")
log(f"configs     : {[c[0] for c in CONFIGS]}")
log(f"label sets  : {LABEL_SCHEMES}")
for tag, p, m in CONFIGS:
    dim = math.comb(m, p)
    log(f"  {tag:6s} -> readout dim {dim:6d} | full-data matrix "
        f"{125973*dim*4/1e9:.2f} GB")

## 2. Data loading & label schemes

In [ ]:
CSV_CANDIDATES = [
    "NSL_KDD_labeled.csv",
    "../../NSL_KDD_labeled.csv",
    "data/NSL_KDD_labeled.csv",
    r"D:\Python\AI\Code\ReservoirComputing\NSL_KDD_labeled.csv",
]
csv_path = next((p for p in CSV_CANDIDATES if Path(p).exists()), None)
assert csv_path, "NSL_KDD_labeled.csv not found - add its path to CSV_CANDIDATES."
log(f"loading {csv_path}")

df = pd.read_csv(csv_path)
categorical_cols = ["protocol_type", "service", "flag"]
target_col       = "label"
numeric_cols     = [c for c in df.columns if c not in categorical_cols + [target_col]]

ohe   = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_cat = ohe.fit_transform(df[categorical_cols]).astype(np.float32)
X_num = df[numeric_cols].to_numpy(dtype=np.float32)
X_all = np.concatenate([X_num, X_cat], axis=1)
raw_label = df[target_col].to_numpy()

log(f"data: {X_all.shape[0]:,} rows x {X_all.shape[1]} features, "
    f"{len(np.unique(raw_label))} distinct labels")


def make_labels(scheme):
    """Return (y_int, class_names) for a label scheme.

    families6 : normal + 5 named attack families + other_attack   (7 classes)
    all23     : every distinct label in the file                  (23 classes)
    """
    if scheme == "families6":
        mapped = np.where(
            raw_label == "normal", "normal",
            np.where(np.isin(raw_label, NAMED_FAMILIES), raw_label, "other_attack"))
        names = ["normal"] + NAMED_FAMILIES + ["other_attack"]
    elif scheme == "all23":
        mapped = raw_label
        counts = pd.Series(raw_label).value_counts()
        names  = ["normal"] + [c for c in counts.index if c != "normal"]
    else:
        raise ValueError(scheme)
    index = {n: i for i, n in enumerate(names)}
    y = np.array([index[v] for v in mapped], dtype=np.int64)
    return y, names


LABELS = {}
for scheme in LABEL_SCHEMES:
    y, names = make_labels(scheme)
    LABELS[scheme] = (y, names)
    counts = pd.Series(y).value_counts().sort_index()
    log(f"[{scheme}] {len(names)} classes")
    for i, n in enumerate(names):
        c = int(counts.get(i, 0))
        flag = "  <-- TINY, treat with care" if c < 50 else ""
        log(f"    {i:2d} {n:<18s} {c:7,d}{flag}")

## 3. Quantum stack — identical to `qorc_zeroday_analysis.ipynb`

`build_quantum_layer` is a direct port of the original, generalised over
`(n_modes, n_photons)`. The structure is unchanged:

* `precircuit` — a fixed random unitary
* `c_var` — one phase shifter per mode, driven by the encoded input (`px1..pxm`)
* `reservoir` — a copy of the pre-circuit unitary
* input state — `n_photons` photons spread evenly across the modes
* readout — unbunched photon-number probabilities, `C(n_modes, n_photons)` of them

`trainable_parameters=[]` — the reservoir never learns. Only the classical head
after the readout is trained.

In [ ]:
def build_quantum_layer(n_modes, n_photons, seed=SEED, device=DEVICE):
    """Build one photonic reservoir. Mirrors the original notebook exactly,
    generalised over (n_modes, n_photons)."""
    np.random.seed(seed)                    # perceval's random_unitary uses global numpy RNG
    U          = pcvl.Matrix.random_unitary(n_modes)
    precircuit = pcvl.Unitary(U)
    reservoir  = precircuit.copy()
    c_var = pcvl.Circuit(n_modes)
    for i in range(n_modes):
        c_var.add(i, pcvl.PS(pcvl.P(f"px{i+1}")))
    qorc_circuit = precircuit // c_var // reservoir

    step = (n_modes - 1) / (n_photons - 1) if n_photons > 1 else 0
    input_state = [0] * n_modes
    for k in range(n_photons):
        input_state[int(round(k * step))] = 1

    ql = ML.QuantumLayer(
        input_size=n_modes, circuit=qorc_circuit,
        trainable_parameters=[], input_parameters=["px"],
        input_state=input_state,
        measurement_strategy=MeasurementStrategy.probs(
            computation_space=ComputationSpace.UNBUNCHED),
        device=device)
    ql.eval()
    return ql


def quantum_features(ql, phases, batch=4096, device=DEVICE):
    """Push encoded phases through the reservoir, in batches, no grad."""
    outs = []
    with torch.no_grad():
        for i in range(0, len(phases), batch):
            xb = torch.tensor(phases[i:i+batch], dtype=torch.float32, device=device)
            outs.append(ql(xb).cpu().numpy())
    return np.concatenate(outs, 0).astype(np.float32)


def to_phase(Z, mu, sd):
    """Smooth tanh squash, no clipping, no wrap-around -> (0, pi).
    Identical to the original notebook."""
    return ((np.pi / 2) * (1 + np.tanh((Z - mu) / (3 * sd)))).astype(np.float32)

## 4. Classical head — GPU softmax regression

A plain multinomial logistic regression, but implemented in torch so it runs on
the GPU. This matters: at `4p24m` the readout is 10,626-dimensional, where
scikit-learn's `LogisticRegression` takes many minutes per fit while this takes
under a second. Using one implementation for **every** config also keeps the
comparison honest — no config gets a different optimiser.

`class_weight="balanced"` is applied by default. With `all23` the largest class
is 20,000x the smallest, and without weighting the head simply never predicts
the tail.

In [ ]:
def closure_value_at(Xt, yt, W, b, w, l2, wsum, resident, chunk, device):
    """Evaluate the training objective at the given weights, without grad.

    Used to record the loss actually reached: torch's LBFGS.step() returns the
    loss from the first closure call (at W=0), which is not informative.
    """
    lossf = torch.nn.CrossEntropyLoss(weight=w, reduction="sum")
    if resident:
        total = float(lossf(Xt @ W + b, yt)) / wsum
    else:
        total = 0.0
        for i in range(0, Xt.shape[0], chunk):
            xb = Xt[i:i+chunk].to(device, non_blocking=True)
            total += float(lossf(xb @ W + b, yt[i:i+chunk]))
            del xb
        total /= wsum
    return total + l2 * float((W * W).sum())


class GPUSoftmaxHead:
    """Multinomial logistic regression trained with L-BFGS on the GPU.

    Equivalent in form to sklearn's LogisticRegression(multi_class='multinomial')
    with L2 regularisation, but fast enough for a 10,626-dim readout.
    """

    # Keep the design matrix GPU-resident only when it comfortably fits. An
    # 8 GB card cannot hold the 4p24m matrix (3.75 GB) alongside the L-BFGS
    # history: measured 7,880 / 8,192 MiB and a stall. Above the threshold the
    # matrix stays in system RAM and the gradient is accumulated over chunks
    # (verified against the resident path to a relative error of 3e-07).
    #
    # Default 0.0 => ALWAYS chunk. The two paths are the same objective, but
    # L-BFGS is path-sensitive: a 1e-7 gradient difference is enough to end at
    # a slightly different point within max_iter. Using one code path for every
    # config keeps the four reservoir sizes exactly comparable, which is the
    # entire point of the sweep. Raise it only if you want speed over
    # comparability on a bigger card.
    GPU_RESIDENT_GB = 0.0

    def __init__(self, n_classes, l2=HEAD_L2, max_iter=HEAD_MAX_ITER,
                 class_weight="balanced", device=DEVICE, chunk=16384):
        self.n_classes, self.l2 = n_classes, l2
        self.max_iter, self.class_weight = max_iter, class_weight
        self.device, self.chunk = device, chunk

    def fit(self, X, y):
        n, d = X.shape
        gb = n * d * 4 / 1e9
        resident = (self.device.type == "cuda") and (gb <= self.GPU_RESIDENT_GB)

        yt = torch.as_tensor(y, dtype=torch.long, device=self.device)
        if self.class_weight == "balanced":
            counts = torch.bincount(yt, minlength=self.n_classes).float()
            w = torch.where(counts > 0, n / (self.n_classes * counts.clamp(min=1)),
                            torch.zeros_like(counts))
        else:
            w = torch.ones(self.n_classes, device=self.device)

        # Xt lives on the GPU only when it comfortably fits; otherwise it stays
        # pinned on the CPU and each chunk is copied across on demand.
        Xt = (torch.as_tensor(X, dtype=torch.float32, device=self.device) if resident
              else torch.from_numpy(np.ascontiguousarray(X)))
        self.W = torch.zeros(d, self.n_classes, device=self.device, requires_grad=True)
        self.b = torch.zeros(self.n_classes,    device=self.device, requires_grad=True)
        lossf  = torch.nn.CrossEntropyLoss(weight=w, reduction="sum")
        opt    = torch.optim.LBFGS([self.W, self.b], max_iter=self.max_iter,
                                   history_size=10, tolerance_grad=1e-7,
                                   line_search_fn="strong_wolfe")
        wsum = float(w[yt].sum())          # normaliser matching reduction="mean"

        def closure():
            opt.zero_grad()
            if resident:
                loss = lossf(Xt @ self.W + self.b, yt) / wsum
                loss = loss + self.l2 * (self.W * self.W).sum()
                loss.backward()
                return loss
            # chunked: accumulate the gradient over slices so peak VRAM is
            # governed by `chunk`, not by the full design matrix
            total = 0.0
            for i in range(0, n, self.chunk):
                xb = Xt[i:i+self.chunk].to(self.device, non_blocking=True)
                yb = yt[i:i+self.chunk]
                part = lossf(xb @ self.W + self.b, yb) / wsum
                part.backward()
                total += float(part)
                del xb, part
            reg = self.l2 * (self.W * self.W).sum()
            reg.backward()
            return torch.tensor(total + float(reg), device=self.device)

        opt.step(closure)
        # NB: LBFGS.step() returns the loss from the FIRST closure evaluation
        # (i.e. at W=0, which is just log(n_classes)), not the converged value.
        # Re-evaluate to record the objective we actually reached.
        with torch.no_grad():
            self.final_loss = float(closure_value_at(Xt, yt, self.W, self.b, w,
                                                     self.l2, wsum, resident,
                                                     self.chunk, self.device))
        self.W = self.W.detach(); self.b = self.b.detach()
        del Xt, yt
        if self.device.type == "cuda":
            torch.cuda.empty_cache()
        return self

    def _logits(self, X, batch=16384):
        outs = []
        with torch.no_grad():
            for i in range(0, len(X), batch):
                xb = torch.as_tensor(X[i:i+batch], dtype=torch.float32, device=self.device)
                outs.append((xb @ self.W + self.b).cpu())
        return torch.cat(outs, 0)

    def predict(self, X):
        return self._logits(X).argmax(1).numpy()

    def predict_proba(self, X):
        return torch.softmax(self._logits(X), dim=1).numpy()

## 5. Metrics & per-experiment runner

Every experiment produces the same record so the configs stay comparable.
`macro_f1` is the headline: with this much class imbalance, plain accuracy is
dominated by `normal` + `neptune` and hides everything interesting.

Each experiment trains **two** heads on the same split:

* `quantum` — the `C(m,p)`-dim reservoir readout
* `dials`   — the `n_modes`-dim PCA input to the reservoir (the classical control)

The `dials` head is the honest baseline: it sees exactly the information that
entered the reservoir, so any difference is attributable to the reservoir itself
rather than to the preprocessing.

In [ ]:
def evaluate(y_true, y_pred, proba, class_names):
    """Full metric bundle for one trained head."""
    p, r, f, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0)
    out = {
        "accuracy":     float(accuracy_score(y_true, y_pred)),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1":     float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1":  float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "per_class": {
            class_names[i]: {"precision": float(p[i]), "recall": float(r[i]),
                             "f1": float(f[i]), "support": int(sup[i])}
            for i in range(len(class_names))},
    }
    # macro OVR AUC, over classes actually present in y_true
    try:
        present = np.unique(y_true)
        if len(present) > 1:
            out["macro_auc_ovr"] = float(roc_auc_score(
                y_true, proba[:, present] / proba[:, present].sum(1, keepdims=True),
                multi_class="ovr", average="macro", labels=present))
    except Exception:
        out["macro_auc_ovr"] = None
    return out


def run_experiment(tag, n_photons, n_modes, scheme, save_artifacts=True):
    """One (reservoir config x label scheme) experiment, end to end."""
    t_start = time.time()
    y_all, class_names = LABELS[scheme]
    n_classes = len(class_names)
    dim = math.comb(n_modes, n_photons)

    gb = X_all.shape[0] * dim * 4 / 1e9
    if gb > MAX_FEATURE_GB:
        raise MemoryError(f"{tag}: feature matrix would be {gb:.1f} GB "
                          f"(limit {MAX_FEATURE_GB} GB)")

    log(f"--- {tag} / {scheme}: {n_classes} classes, readout dim {dim} Fock states, "
        f"{gb:.2f} GB ---")

    # ---- split (stratified on the fine-grained label so rare classes appear
    #      in both halves where possible) -------------------------------------
    counts = pd.Series(y_all).value_counts()
    strat  = y_all if counts.min() >= 2 else None
    if strat is None:
        log("class with <2 rows present; falling back to unstratified split", "WARN")
    idx_tr, idx_te = train_test_split(
        np.arange(len(y_all)), test_size=TEST_SIZE, random_state=SEED, stratify=strat)
    y_tr, y_te = y_all[idx_tr], y_all[idx_te]

    # ---- front-end: fitted on TRAIN only ------------------------------------
    t0 = time.time()
    scaler = StandardScaler().fit(X_all[idx_tr])
    A = scaler.transform(X_all[idx_tr]).astype(np.float32)
    B = scaler.transform(X_all[idx_te]).astype(np.float32)
    pca = PCA(n_components=n_modes, random_state=SEED).fit(A)
    Za, Zb = pca.transform(A), pca.transform(B)
    mu, sd = Za.mean(0), Za.std(0) + 1e-8
    del A, B
    log(f"  front-end fitted ({time.time()-t0:.1f}s), "
        f"PCA explains {pca.explained_variance_ratio_.sum():.1%} of variance")

    # ---- quantum features ---------------------------------------------------
    t0 = time.time()
    ql = build_quantum_layer(n_modes, n_photons)
    Qa = quantum_features(ql, to_phase(Za, mu, sd))
    Qb = quantum_features(ql, to_phase(Zb, mu, sd))
    t_quantum = time.time() - t0
    log(f"  Fock-state readout: {Qa.shape[0]:,}+{Qb.shape[0]:,} rows x {Qa.shape[1]} "
        f"in {t_quantum:.1f}s ({t_quantum/(len(Qa)+len(Qb))*1e3:.3f} ms/row)")

    # Standardise IN PLACE. At 10,626 dims a naive
    # `qs.transform(Qa).astype(np.float32)` allocates two extra full copies and
    # pushes RSS past 11 GB, which sends the machine into swap. Doing the
    # arithmetic in place keeps the peak at one matrix.
    def standardize_inplace(Q):
        mean = Q.mean(0, dtype=np.float64).astype(np.float32)
        std  = Q.std(0, dtype=np.float64).astype(np.float32)
        std[std < 1e-8] = 1.0            # constant columns -> leave at zero
        Q -= mean
        Q /= std
        return mean, std

    q_mean, q_std = standardize_inplace(Qa)
    Qb -= q_mean; Qb /= q_std            # test set uses TRAIN statistics

    Ra = Za.astype(np.float32, copy=True); Rb = Zb.astype(np.float32, copy=True)
    r_mean, r_std = standardize_inplace(Ra)
    Rb -= r_mean; Rb /= r_std

    # ---- heads --------------------------------------------------------------
    rec = {"config": tag, "n_photons": n_photons, "n_modes": n_modes,
           "readout_dim": dim, "scheme": scheme, "n_classes": n_classes,
           "class_names": class_names, "n_train": len(idx_tr), "n_test": len(idx_te),
           "pca_explained_var": float(pca.explained_variance_ratio_.sum()),
           "t_quantum_s": t_quantum, "heads": {}}

    for head_name, Xtr, Xte in [("quantum", Qa, Qb), ("dials", Ra, Rb)]:
        t0 = time.time()
        head = GPUSoftmaxHead(n_classes).fit(Xtr, y_tr)
        pred = head.predict(Xte); proba = head.predict_proba(Xte)
        m = evaluate(y_te, pred, proba, class_names)
        m["fit_time_s"]  = time.time() - t0
        m["final_loss"]  = head.final_loss
        m["input_dim"]   = int(Xtr.shape[1])
        rec["heads"][head_name] = m
        log(f"  {head_name:8s} dim={Xtr.shape[1]:5d} acc={m['accuracy']:.4f} "
            f"bal_acc={m['balanced_acc']:.4f} macro_F1={m['macro_f1']:.4f} "
            f"({m['fit_time_s']:.1f}s)")

        if save_artifacts:
            np.save(RUN_DIR / "artifacts" / f"cm_{tag}_{scheme}_{head_name}.npy",
                    confusion_matrix(y_te, pred, labels=range(n_classes)))
        del head, proba
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    q, d = rec["heads"]["quantum"], rec["heads"]["dials"]
    rec["gain_macro_f1"]     = q["macro_f1"]     - d["macro_f1"]
    rec["gain_balanced_acc"] = q["balanced_acc"] - d["balanced_acc"]
    rec["gain_accuracy"]     = q["accuracy"]     - d["accuracy"]
    rec["total_time_s"]      = time.time() - t_start
    log(f"  GAIN quantum-dials: macro_F1 {rec['gain_macro_f1']:+.4f} | "
        f"bal_acc {rec['gain_balanced_acc']:+.4f} "
        f"(total {rec['total_time_s']:.1f}s)")

    del Qa, Qb, Ra, Rb, Za, Zb, ql
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return rec

## 6. Smoke test — validate everything before the long run

This runs the **entire pipeline** on a 6,000-row subsample with the smallest
config, plus a set of assertions on the pieces that are easy to get silently
wrong. It should finish in well under a minute.

**Do not start the overnight run until this cell prints `SMOKE TEST PASSED`.**

In [ ]:
def smoke_test():
    global X_all, LABELS
    ok, checks = True, []

    def check(name, cond, detail=""):
        nonlocal ok
        ok &= bool(cond)
        checks.append((name, bool(cond), detail))
        log(f"  [{'PASS' if cond else 'FAIL'}] {name} {detail}",
            "INFO" if cond else "ERROR")

    log("=== SMOKE TEST ===")

    # 1. reservoir shape & determinism
    ql = build_quantum_layer(12, 3, seed=SEED)
    x  = np.random.uniform(0, np.pi, size=(64, 12)).astype(np.float32)
    f1 = quantum_features(ql, x)
    check("readout dim == C(12,3)", f1.shape[1] == math.comb(12, 3),
          f"got {f1.shape[1]}, expected {math.comb(12,3)}")
    check("probabilities in [0,1]", (f1 >= -1e-6).all() and (f1 <= 1+1e-6).all(),
          f"range [{f1.min():.3e}, {f1.max():.3e}]")
    check("rows sum to ~1", np.allclose(f1.sum(1), 1.0, atol=1e-3),
          f"sums in [{f1.sum(1).min():.4f}, {f1.sum(1).max():.4f}]")
    f2 = quantum_features(build_quantum_layer(12, 3, seed=SEED), x)
    check("reservoir reproducible for a fixed seed", np.allclose(f1, f2, atol=1e-6))
    f3 = quantum_features(build_quantum_layer(12, 3, seed=SEED + 1), x)
    check("different seed -> different reservoir", not np.allclose(f1, f3, atol=1e-6))

    # 2. every config builds and gives the right dim
    for tag, p, m in CONFIGS:
        try:
            q = build_quantum_layer(m, p)
            d = quantum_features(q, np.random.uniform(0, np.pi, (8, m)).astype(np.float32))
            check(f"config {tag} builds, dim {math.comb(m,p)}",
                  d.shape[1] == math.comb(m, p), f"got {d.shape[1]}")
            del q, d
        except Exception as e:
            check(f"config {tag} builds", False, f"{type(e).__name__}: {e}")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    # 3. encoding is bounded, monotonic, and non-degenerate over the real range.
    #    NB: tanh saturates in float32 past about |z/3sd| > 9, so the phase
    #    reaches exactly 0 / pi at extreme inputs. That is graceful saturation,
    #    not the hard clipping the tanh encoding was introduced to avoid: order
    #    is preserved everywhere and nothing wraps around.
    Z  = np.linspace(-50, 50, 500).reshape(-1, 1)
    ph = to_phase(Z, np.array([0.0]), np.array([1.0]))
    check("phase bounded in [0, pi]", (ph >= 0).all() and (ph <= np.pi).all(),
          f"range [{ph.min():.4f}, {ph.max():.4f}]")
    check("phase monotonic (never decreases, so order is preserved)",
          (np.diff(ph.ravel()) >= 0).all())
    # over +/-4 sd -- where essentially all real data lives -- it must be strict
    Zc = np.linspace(-12, 12, 500).reshape(-1, 1)
    phc = to_phase(Zc, np.array([0.0]), np.array([1.0]))
    check("phase strictly increasing over +/-4 sd (no clipping on real data)",
          (np.diff(phc.ravel()) > 0).all(),
          f"range [{phc.min():.4f}, {phc.max():.4f}]")
    check("encoding spreads real data across the range (not collapsed)",
          phc.std() > 0.3, f"std={phc.std():.3f}")

    # 4. label schemes are consistent
    for scheme in LABEL_SCHEMES:
        y, names = LABELS[scheme]
        check(f"[{scheme}] labels cover all rows", len(y) == len(X_all))
        check(f"[{scheme}] label ids within range",
              y.min() >= 0 and y.max() < len(names), f"{len(names)} classes")
        check(f"[{scheme}] normal is class 0", names[0] == "normal")
    y6, n6 = LABELS["families6"]
    check("[families6] exactly 7 classes", len(n6) == 7, f"got {len(n6)}")
    check("[families6] no class under 800 rows",
          pd.Series(y6).value_counts().min() >= 800,
          f"min={pd.Series(y6).value_counts().min()}")

    # 5. the head learns something on a trivially separable problem
    rng = np.random.default_rng(0)
    Xs  = np.vstack([rng.normal(m_, 0.5, (200, 5)) for m_ in (-3, 0, 3)]).astype(np.float32)
    ys  = np.repeat([0, 1, 2], 200)
    h   = GPUSoftmaxHead(3, max_iter=100).fit(Xs, ys)
    acc = accuracy_score(ys, h.predict(Xs))
    check("head solves a separable 3-class toy problem", acc > 0.97, f"acc={acc:.3f}")

    # 6. full pipeline on a subsample
    X_full, L_full = X_all, LABELS
    try:
        sub = np.random.default_rng(SEED).choice(len(X_full), 6000, replace=False)
        X_all = X_full[sub]
        LABELS = {"families6": (L_full["families6"][0][sub], L_full["families6"][1])}
        r = run_experiment("3p12m", 3, 12, "families6", save_artifacts=False)
        check("end-to-end experiment returns metrics",
              0 <= r["heads"]["quantum"]["macro_f1"] <= 1,
              f"macro_F1={r['heads']['quantum']['macro_f1']:.3f}")
        check("result record is JSON-serialisable", bool(json.dumps(r)))
    except Exception as e:
        check("end-to-end experiment", False, f"{type(e).__name__}: {e}")
        traceback.print_exc()
    finally:
        X_all, LABELS = X_full, L_full

    # 7. enough disk for the log + artifacts
    import shutil
    free_gb = shutil.disk_usage(RUN_DIR).free / 1e9
    check("at least 1 GB free disk", free_gb > 1.0, f"{free_gb:.1f} GB free")

    n_fail = sum(1 for _, c, _ in checks if not c)
    log("=" * 60)
    if ok:
        log(f"SMOKE TEST PASSED ({len(checks)} checks) - safe to start the full run")
    else:
        log(f"SMOKE TEST FAILED - {n_fail}/{len(checks)} checks failed. "
            f"Fix before running overnight.", "ERROR")
    log("=" * 60)
    return ok

SMOKE_OK = smoke_test()

## 7. The full sweep — crash-safe, resumable

Every experiment is checkpointed to `results.jsonl` **the moment it finishes**,
one JSON object per line, flushed and `fsync`-ed to disk. If the machine dies
overnight, rerunning this cell skips everything already recorded and resumes
where it stopped.

A failing experiment is logged with its traceback and the sweep continues — one
bad config cannot take down the whole night.

In [ ]:
RESULTS_PATH = RUN_DIR / "results.jsonl"

# Resume support: reuse the newest run dir that already has partial results.
def find_resumable():
    for d in sorted(RUN_ROOT.iterdir(), reverse=True):
        if d.is_dir() and (d / "results.jsonl").exists() and d != RUN_DIR:
            return d
    return None

RESUME_FROM = None          # <- set to a Path to resume a previous run, or leave None
if RESUME_FROM is not None:
    RESULTS_PATH = Path(RESUME_FROM) / "results.jsonl"
    log(f"RESUMING into {RESULTS_PATH}", "WARN")

def load_done():
    done = {}
    if RESULTS_PATH.exists():
        with open(RESULTS_PATH, encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                    done[(r["config"], r["scheme"])] = r
                except json.JSONDecodeError:
                    log("skipping a corrupt results line", "WARN")
    return done

def append_result(rec):
    with open(RESULTS_PATH, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(rec) + "\n")
        fh.flush()
        os.fsync(fh.fileno())

if not SMOKE_OK:
    raise RuntimeError("Smoke test failed - refusing to start the full sweep. "
                       "Fix the failures reported above first.")

done  = load_done()
plan  = [(tag, p, m, s) for (tag, p, m) in CONFIGS for s in LABEL_SCHEMES]
todo  = [j for j in plan if (j[0], j[3]) not in done]

log("=" * 70)
log(f"SWEEP: {len(plan)} experiments, {len(done)} already done, {len(todo)} to run")
log(f"results -> {RESULTS_PATH}")
log("=" * 70)

t_sweep = time.time()
for i, (tag, p, m, scheme) in enumerate(todo, 1):
    log(f"[{i}/{len(todo)}] {tag} / {scheme}   (elapsed {(time.time()-t_sweep)/60:.1f} min)")
    try:
        rec = run_experiment(tag, p, m, scheme)
        rec["run_id"], rec["timestamp"] = RUN_ID, datetime.now().isoformat()
        append_result(rec)
        done[(tag, scheme)] = rec
        log(f"[{i}/{len(todo)}] saved.")
    except Exception as e:
        log(f"[{i}/{len(todo)}] {tag}/{scheme} FAILED: {type(e).__name__}: {e}", "ERROR")
        with open(RUN_DIR / "errors.log", "a", encoding="utf-8") as fh:
            fh.write(f"\n=== {tag}/{scheme} @ {datetime.now().isoformat()} ===\n")
            traceback.print_exc(file=fh)
            fh.flush()
        traceback.print_exc()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

log("=" * 70)
log(f"SWEEP COMPLETE in {(time.time()-t_sweep)/60:.1f} min - "
    f"{len(done)}/{len(plan)} experiments on disk")
log("=" * 70)

---
# Part II — Result investigation

Everything below reads from `results.jsonl`, so it can be run in a **fresh
kernel** the morning after without recomputing anything. Point `INSPECT_PATH`
at the run you want to analyse.

## 8. Load results

In [ ]:
# Analyse the current run by default; set explicitly to inspect an older one.
INSPECT_PATH = RESULTS_PATH        # e.g. Path("qorc_sup_runs/20260726_013000/results.jsonl")

rows = []
with open(INSPECT_PATH, encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            rows.append(json.loads(line))
print(f"loaded {len(rows)} experiments from {INSPECT_PATH}")

CONFIG_ORDER = [c[0] for c in CONFIGS]

# Shared handles used by every analysis cell below. Defined here so the cells
# can be re-run individually in any order once this one has run.
by_key  = {(r["config"], r["scheme"]): r for r in rows}
schemes = sorted({r["scheme"] for r in rows})
art     = Path(INSPECT_PATH).parent / "artifacts"

flat = []
for r in rows:
    for head in ("quantum", "dials"):
        h = r["heads"][head]
        flat.append({
            "config": r["config"], "scheme": r["scheme"], "head": head,
            "readout_dim": r["readout_dim"], "input_dim": h["input_dim"],
            "n_photons": r["n_photons"], "n_modes": r["n_modes"],
            "n_classes": r["n_classes"],
            "accuracy": h["accuracy"], "balanced_acc": h["balanced_acc"],
            "macro_f1": h["macro_f1"], "weighted_f1": h["weighted_f1"],
            "macro_auc": h.get("macro_auc_ovr"),
            "fit_time_s": h["fit_time_s"], "t_quantum_s": r["t_quantum_s"],
        })
D = pd.DataFrame(flat)
D["config"] = pd.Categorical(D["config"], CONFIG_ORDER, ordered=True)
D = D.sort_values(["scheme", "config", "head"]).reset_index(drop=True)
D.round(4)

### 8.1 Headline table — quantum vs classical control, per config

In [ ]:
for scheme in sorted(D["scheme"].unique()):
    sub = D[D["scheme"] == scheme]
    piv = sub.pivot_table(index="config",
                          columns="head",
                          values=["macro_f1", "balanced_acc", "accuracy"],
                          observed=True)
    piv = piv.reindex([c for c in CONFIG_ORDER if c in piv.index])
    out = pd.DataFrame(index=piv.index)
    for met in ("macro_f1", "balanced_acc", "accuracy"):
        out[f"{met}_q"]    = piv[(met, "quantum")]
        out[f"{met}_dials"] = piv[(met, "dials")]
        out[f"{met}_gain"] = piv[(met, "quantum")] - piv[(met, "dials")]
    dims = sub.drop_duplicates("config").set_index("config")["readout_dim"]
    out.insert(0, "readout_dim", dims.reindex(out.index))
    print(f"\n===== {scheme} =====")
    print(out.round(4).to_string())

### 8.2 Does more reservoir capacity help?

The core question this sweep exists to answer. If the quantum curve rises with
readout dimension while the classical control stays flat, the reservoir is
contributing something real. If both rise together, it is the PCA dials doing
the work.

In [ ]:
fig, axes = plt.subplots(1, len(schemes), figsize=(6.2*len(schemes), 4.6), squeeze=False)
for ax, scheme in zip(axes[0], schemes):
    sub = D[D["scheme"] == scheme]
    for head, color, mk in [("quantum", "#2E7D32", "o"), ("dials", "#5B8FF9", "s")]:
        s = sub[sub["head"] == head].sort_values("readout_dim")
        ax.plot(s["readout_dim"], s["macro_f1"], marker=mk, color=color,
                label=f"{head} head", lw=2, ms=8)
        for _, r in s.iterrows():
            ax.annotate(r["config"], (r["readout_dim"], r["macro_f1"]),
                        textcoords="offset points", xytext=(0, 9),
                        ha="center", fontsize=7.5, color=color)
    ax.set_xscale("log")
    ax.set_xlabel("readout dimension  C(modes, photons)   [log scale]")
    ax.set_ylabel("macro F1")
    ax.set_title(f"{scheme}: capacity vs macro F1")
    ax.grid(alpha=0.3); ax.legend()
fig.suptitle("Does a bigger reservoir classify attacks better?", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

### 8.3 Quantum gain per config — quantum head minus classical control

In [ ]:
fig, axes = plt.subplots(1, len(schemes), figsize=(6.2*len(schemes), 4.4),
                         squeeze=False, sharey=True)
for ax, scheme in zip(axes[0], schemes):
    g = []
    for cfg in CONFIG_ORDER:
        s = D[(D["scheme"] == scheme) & (D["config"] == cfg)]
        if len(s) == 2:
            q = s[s["head"] == "quantum"].iloc[0]
            d = s[s["head"] == "dials"].iloc[0]
            g.append((cfg, q["macro_f1"] - d["macro_f1"],
                      q["balanced_acc"] - d["balanced_acc"]))
    if not g:
        continue
    labels = [x[0] for x in g]
    x = np.arange(len(g)); w = 0.38
    ax.bar(x - w/2, [x_[1] for x_ in g], w, label="macro F1 gain",
           color=["#2E7D32" if x_[1] > 0 else "#C62828" for x_ in g])
    ax.bar(x + w/2, [x_[2] for x_ in g], w, label="balanced acc gain",
           color=["#81C784" if x_[2] > 0 else "#EF9A9A" for x_ in g])
    ax.axhline(0, color="k", lw=1)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_title(f"{scheme}"); ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=8)
axes[0][0].set_ylabel("quantum - classical control")
fig.suptitle("Reservoir contribution over its own input", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

### 8.4 Per-class F1 — where does the reservoir actually change things?

An aggregate number hides the interesting behaviour. This is the table to read
when writing up: it shows which specific attack families benefit.

In [ ]:
for scheme in schemes:
    names = next(r["class_names"] for r in rows if r["scheme"] == scheme)
    # for all23 keep the classes big enough to mean anything
    keep = names if scheme == "families6" else [
        n for n in names
        if any(by_key[(c, scheme)]["heads"]["quantum"]["per_class"][n]["support"] >= 20
               for c in CONFIG_ORDER if (c, scheme) in by_key)]
    tbl = {}
    for cfg in CONFIG_ORDER:
        if (cfg, scheme) not in by_key:
            continue
        r = by_key[(cfg, scheme)]
        for head in ("quantum", "dials"):
            pc = r["heads"][head]["per_class"]
            tbl[f"{cfg}_{head[0]}"] = {n: pc[n]["f1"] for n in keep}
    T = pd.DataFrame(tbl)
    T.insert(0, "support", [by_key[(CONFIG_ORDER[0], scheme)]["heads"]["quantum"]
                            ["per_class"][n]["support"] for n in keep])
    print(f"\n===== per-class F1 - {scheme}  (_q = quantum, _d = dials) =====")
    print(T.round(3).to_string())

In [ ]:
# Heatmap: per-class F1 gain (quantum - dials), classes x configs
for scheme in schemes:
    names = next(r["class_names"] for r in rows if r["scheme"] == scheme)
    cfgs  = [c for c in CONFIG_ORDER if (c, scheme) in by_key]
    keep  = [n for n in names
             if by_key[(cfgs[0], scheme)]["heads"]["quantum"]["per_class"][n]["support"] >= 20]
    M = np.zeros((len(keep), len(cfgs)))
    for j, cfg in enumerate(cfgs):
        r = by_key[(cfg, scheme)]
        for i, n in enumerate(keep):
            M[i, j] = (r["heads"]["quantum"]["per_class"][n]["f1"]
                       - r["heads"]["dials"]["per_class"][n]["f1"])
    lim = max(0.05, np.abs(M).max())
    fig, ax = plt.subplots(figsize=(1.5*len(cfgs)+3.5, 0.42*len(keep)+2))
    im = ax.imshow(M, cmap="RdYlGn", vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(range(len(cfgs))); ax.set_xticklabels(cfgs)
    ax.set_yticks(range(len(keep))); ax.set_yticklabels(keep, fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i,j]:+.2f}", ha="center", va="center", fontsize=7.5)
    ax.set_title(f"per-class F1 gain (quantum - dials) - {scheme}\n"
                 f"green = reservoir helps this class", fontsize=11)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="F1 gain")
    plt.tight_layout(); plt.show()

### 8.5 Confusion matrices — what gets mistaken for what

Row-normalised, so each row reads as "of the true class, what fraction went
where". The `normal` row and column matter most operationally: `normal`
misclassified as an attack is a false alarm, and an attack landing in `normal`
is a miss.

In [ ]:
SHOW_SCHEME = "families6"      # switch to "all23" for the fine-grained version
names = next(r["class_names"] for r in rows if r["scheme"] == SHOW_SCHEME)
cfgs  = [c for c in CONFIG_ORDER if (c, SHOW_SCHEME) in by_key]

fig, axes = plt.subplots(2, len(cfgs), figsize=(4.1*len(cfgs), 8.4), squeeze=False)
for j, cfg in enumerate(cfgs):
    for i, head in enumerate(["quantum", "dials"]):
        ax = axes[i][j]
        f = art / f"cm_{cfg}_{SHOW_SCHEME}_{head}.npy"
        if not f.exists():
            ax.axis("off"); ax.set_title(f"{cfg}/{head}\n(missing)", fontsize=9); continue
        cm = np.load(f).astype(float)
        cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
        ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names, rotation=90, fontsize=6.5)
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names if j == 0 else [], fontsize=6.5)
        for a in range(len(names)):
            for b in range(len(names)):
                if cmn[a, b] > 0.01:
                    ax.text(b, a, f"{cmn[a,b]:.2f}", ha="center", va="center",
                            fontsize=5.5, color="white" if cmn[a, b] > 0.5 else "black")
        ax.set_title(f"{cfg} / {head}", fontsize=9)
fig.suptitle(f"Row-normalised confusion matrices - {SHOW_SCHEME}", y=1.005, fontsize=13)
plt.tight_layout(); plt.show()

### 8.6 Operational view — false alarms and misses

Two numbers a security team actually cares about, derived from the confusion
matrix regardless of how many attack classes there are:

* **false-alarm rate** — normal traffic predicted as some attack
* **miss rate** — attack traffic predicted as normal
* **attack-type accuracy** — of the attacks correctly identified *as* attacks,
  how often the specific family is right (i.e. is the triage useful?)

In [ ]:
op = []
for scheme in schemes:
    names = next(r["class_names"] for r in rows if r["scheme"] == scheme)
    for cfg in [c for c in CONFIG_ORDER if (c, scheme) in by_key]:
        for head in ("quantum", "dials"):
            f = art / f"cm_{cfg}_{scheme}_{head}.npy"
            if not f.exists():
                continue
            cm = np.load(f).astype(float)
            n_norm  = cm[0].sum()
            n_atk   = cm[1:].sum()
            false_alarm = cm[0, 1:].sum() / max(n_norm, 1)
            miss        = cm[1:, 0].sum() / max(n_atk, 1)
            caught      = cm[1:, 1:]
            type_acc    = np.trace(caught) / max(caught.sum(), 1)
            op.append({"scheme": scheme, "config": cfg, "head": head,
                       "false_alarm": false_alarm, "miss_rate": miss,
                       "attack_type_acc": type_acc})
OP = pd.DataFrame(op)
if len(OP):
    OP["config"] = pd.Categorical(OP["config"], CONFIG_ORDER, ordered=True)
    OP = OP.sort_values(["scheme", "config", "head"])
    for scheme in schemes:
        print(f"\n===== operational metrics - {scheme} =====")
        print(OP[OP["scheme"] == scheme]
              .drop(columns="scheme").to_string(index=False,
                    formatters={"false_alarm": "{:.4f}".format,
                                "miss_rate": "{:.4f}".format,
                                "attack_type_acc": "{:.4f}".format}))

### 8.7 Cost — is the biggest reservoir worth what it costs?

In [ ]:
cost = D[D["head"] == "quantum"].drop_duplicates(["config", "scheme"]).copy()
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.4))
for scheme, mk in zip(schemes, ["o", "s"]):
    s = cost[cost["scheme"] == scheme].sort_values("readout_dim")
    ax[0].plot(s["t_quantum_s"], s["macro_f1"], marker=mk, ms=9, lw=1.6, label=scheme)
    for _, r in s.iterrows():
        ax[0].annotate(r["config"], (r["t_quantum_s"], r["macro_f1"]),
                       textcoords="offset points", xytext=(0, 9), ha="center", fontsize=7.5)
    ax[1].plot(s["readout_dim"], s["t_quantum_s"], marker=mk, ms=9, lw=1.6, label=scheme)
ax[0].set_xlabel("quantum feature time (s, whole dataset)"); ax[0].set_ylabel("macro F1")
ax[0].set_title("Accuracy bought per second of reservoir time")
ax[1].set_xlabel("readout dim"); ax[1].set_ylabel("quantum feature time (s)")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_title("Reservoir cost scaling")
for a in ax:
    a.grid(alpha=0.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 8.8 Summary — what the sweep concluded

Auto-generated from the results so the written claim always matches the numbers
on disk. Read the caveats: they are the difference between an honest result and
an overclaim.

In [ ]:
lines = ["", "=" * 72, "SUMMARY", "=" * 72]
for scheme in schemes:
    lines.append(f"\n--- {scheme} ---")
    sub = D[D["scheme"] == scheme]
    best_q = sub[sub["head"] == "quantum"].nlargest(1, "macro_f1")
    best_d = sub[sub["head"] == "dials"].nlargest(1, "macro_f1")
    if len(best_q) and len(best_d):
        bq, bd = best_q.iloc[0], best_d.iloc[0]
        lines.append(f"best quantum : {bq['config']} (dim {bq['readout_dim']}) "
                     f"macro_F1={bq['macro_f1']:.4f}  bal_acc={bq['balanced_acc']:.4f}")
        lines.append(f"best control : {bd['config']} (dim {bd['input_dim']}) "
                     f"macro_F1={bd['macro_f1']:.4f}  bal_acc={bd['balanced_acc']:.4f}")
        delta = bq["macro_f1"] - bd["macro_f1"]
        verdict = ("reservoir ahead of its own input" if delta > 0.01 else
                   "reservoir behind its own input"   if delta < -0.01 else
                   "reservoir indistinguishable from its own input")
        lines.append(f"  -> best-vs-best macro F1 delta = {delta:+.4f}  ({verdict})")

    gains = []
    for cfg in CONFIG_ORDER:
        s = sub[sub["config"] == cfg]
        if len(s) == 2:
            g = (s[s["head"] == "quantum"].iloc[0]["macro_f1"]
                 - s[s["head"] == "dials"].iloc[0]["macro_f1"])
            gains.append((cfg, g))
    if gains:
        pos = sum(1 for _, g in gains if g > 0)
        lines.append(f"  per-config gain: {pos}/{len(gains)} configs favour quantum "
                     + " ".join(f"[{c}:{g:+.3f}]" for c, g in gains))
        qs = sub[sub["head"] == "quantum"].sort_values("readout_dim")
        if len(qs) >= 2:
            trend = qs.iloc[-1]["macro_f1"] - qs.iloc[0]["macro_f1"]
            lines.append(f"  capacity trend (smallest -> largest reservoir): "
                         f"{trend:+.4f} macro F1")

lines += ["", "-" * 72, "CAVEATS", "-" * 72,
          "* Single seed (42) per config. The reservoir unitary is random; the",
          "  original zero-day notebook saw std up to 0.17 across seeds on one",
          "  arm. A capacity trend smaller than that is not yet established -",
          "  rerun with 2-3 seeds per config before claiming it.",
          "* This is CLOSED-SET classification on known attack families. It says",
          "  nothing about zero-day performance; that is the novelty head's job.",
          "* all23 contains classes with 2-11 rows. Their F1 is noise, not signal.",
          "  Quote families6 for headline numbers.",
          "* The 'dials' control shares the reservoir's own PCA front-end, so it",
          "  is a tight control: any gain is attributable to the reservoir, not",
          "  to preprocessing."]
summary = "\n".join(lines)
print(summary)
(Path(INSPECT_PATH).parent / "summary.txt").write_text(summary, encoding="utf-8")
print(f"\n(saved to {Path(INSPECT_PATH).parent / 'summary.txt'})")

### 8.9 Pitch figure — attack classification accuracy

The deck version of the capacity question. The stored `all23` confusion matrix
is folded down to **normal + the 4 canonical KDD families**, and accuracy is read
off that folded matrix — so a `neptune` row predicted as `smurf` counts as
correct (both are DoS), which is the claim the slide actually makes. Note this is
*not* the `families6` scheme: that one is normal + 5 frequent labels + a catch-all,
which is a different question.

Bars are grouped by mode count so 3 vs 4 photons sit side by side. The dashed
line is the mean accuracy of the four classical baselines from
`classical_baselines.json`, on the same 5-class problem.

Writes `pitch_figures/12_supervised_accuracy.png`. This is the same figure
`qorc_pitch_figures.ipynb` produces; it lives here too so the sweep is
self-contained and the figure can never drift from the run that made it.


In [ ]:
# --------------------------------------------------------------- pitch style
# Self-contained: needs `art`, `rows`, `CONFIG_ORDER` from cell 8 and
# `raw_label` from cell 2.
import matplotlib as mpl
from matplotlib.ticker import PercentFormatter

PINK      = "#6D28E0"     # quantum -- the hero colour (violet)
PINK_SOFT = "#A97CF0"     # quantum, secondary series
DIM_DARK  = "#8A8A8A"     # classical reference -- deliberately quiet
INK       = "#3C3C46"
GRID      = "#EAEAF2"

PITCH_RC = {
    "figure.dpi": 120, "savefig.dpi": 300,
    "savefig.bbox": "tight", "savefig.facecolor": "white",
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelsize": 11, "axes.edgecolor": "#CCCCCC",
    "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "legend.frameon": False,
}

FIGDIR = Path("pitch_figures")
FIGDIR.mkdir(exist_ok=True)

def save_pitch(fig, name):
    p = FIGDIR / f"{name}.png"
    fig.savefig(p)
    print(f"  saved {p}")
    return p

# ---- canonical NSL-KDD / KDD'99 taxonomy --------------------------------
# Deliberately NOT the `families6` scheme: that one is normal + 5 frequent
# labels + a catch-all. This is the textbook 4-family grouping, verified by
# assertion rather than trusted.
FAMILY_OF = {}
for fam, labels in {
    "DoS":   ["neptune", "smurf", "back", "teardrop", "pod", "land",
              "apache2", "udpstorm", "processtable", "mailbomb"],
    "Probe": ["satan", "ipsweep", "portsweep", "nmap", "mscan", "saint"],
    "R2L":   ["warezclient", "guess_passwd", "warezmaster", "imap", "ftp_write",
              "multihop", "phf", "spy", "sendmail", "named", "snmpgetattack",
              "snmpguess", "xlock", "xsnoop", "worm"],
    "U2R":   ["buffer_overflow", "rootkit", "loadmodule", "perl", "sqlattack",
              "xterm", "ps"],
}.items():
    for l in labels:
        FAMILY_OF[l] = fam

FAM_ORDER = ["DoS", "Probe", "R2L", "U2R"]
_missing = [l for l in set(raw_label) if l != "normal" and l not in FAMILY_OF]
assert not _missing, f"labels with no family mapping: {_missing}"


def family_confusion(config, head, scheme="all23"):
    """Fold the stored 23x23 confusion matrix into normal + 4 families.

    Exact, not approximate: it re-aggregates the same predictions the sweep
    already made, so no model is re-run here.
    """
    rec = next(r for r in rows
               if r["config"] == config and r["scheme"] == scheme)
    names = rec["class_names"]
    cm = np.load(art / f"cm_{config}_{scheme}_{head}.npy").astype(float)

    group = ["normal"] + FAM_ORDER
    idx = [group.index("normal" if n == "normal" else FAMILY_OF[n]) for n in names]
    F = np.zeros((len(group), len(group)))
    for i in range(len(names)):
        for j in range(len(names)):
            F[idx[i], idx[j]] += cm[i, j]
    return F


def family_accuracy(config, head):
    """Overall accuracy over normal + the 4 families (diagonal / total)."""
    F = family_confusion(config, head)
    return np.trace(F) / F.sum()

sup_acc = {"quantum": {}, "dials": {}}
for c in CONFIG_ORDER:
    for head in ("quantum", "dials"):
        try:
            sup_acc[head][c] = family_accuracy(c, head)
        except (StopIteration, FileNotFoundError):
            pass

print("=== accuracy over normal + 4 KDD families (from the all23 CM) ===")
print(pd.DataFrame(sup_acc).round(4).to_string())

# ------------------------------------------------------------------ grouping
plotted = sorted(sup_acc["quantum"])
MODES    = sorted({int(c.split("p")[1].rstrip("m")) for c in plotted})
PHOTONS  = sorted({int(c.split("p")[0]) for c in plotted})
PH_COLOR = {3: PINK_SOFT, 4: PINK}        # 4 photons = the deeper violet
gx       = np.arange(len(MODES))
GW       = 0.34                            # bar width within a group

def draw_groups(ax, lookup, fmt="{:.1f}%", scale=100, fontsize=11):
    """Grouped bars + value labels. Returns the flat list of plotted values."""
    vals = []
    for k, p in enumerate(PHOTONS):
        v = [lookup.get(f"{p}p{m}m", np.nan) for m in MODES]
        vals += [q for q in v if np.isfinite(q)]
        off = (k - (len(PHOTONS) - 1) / 2) * GW
        ax.bar(gx + off, v, GW, color=PH_COLOR.get(p, PINK), zorder=3,
               label=f"{p} photons")
        for xi, q in zip(gx + off, v):
            if np.isfinite(q):
                ax.annotate(fmt.format(q * scale), (xi, q),
                            textcoords="offset points", xytext=(0, 6),
                            ha="center", fontsize=fontsize,
                            fontweight="bold", color=INK, zorder=5)
    ax.set_xticks(gx)
    ax.set_xticklabels([f"{m} modes\n{math.comb(m, 3):,} / {math.comb(m, 4):,} Fock states"
                        for m in MODES], fontsize=9)
    return vals

# ------------------------------------------------- classical reference line
BASE_PATH = Path("classical_baselines.json")
assert BASE_PATH.exists(), (
    "classical_baselines.json missing - run classical_baselines.py first")
BASE = json.loads(BASE_PATH.read_text(encoding="utf-8"))
REF_MODELS = ["Random Forest", "Random Forest (PCA-24)", "MLP (100,)",
              "Logistic Regression"]
avg_classical = float(np.mean([BASE["supervised"][m]["accuracy"]
                               for m in REF_MODELS]))

with mpl.rc_context(PITCH_RC):
    fig, ax = plt.subplots(figsize=(8.6, 5.6))
    vals = draw_groups(ax, sup_acc["quantum"])

    ax.axhline(avg_classical, color=DIM_DARK, lw=1.6, ls="--", zorder=2)
    ax.annotate(f"Average classical baseline \u2248 {avg_classical*100:.1f}%",
                xy=(gx[-1] + 0.42, avg_classical), xytext=(0, 6),
                textcoords="offset points", ha="right", va="bottom",
                fontsize=8.5, color=DIM_DARK)

    lo, hi = min(vals + [avg_classical]), max(vals + [avg_classical])
    pad = (hi - lo) * 0.32
    ax.set_ylim(lo - pad, min(hi + pad, 1.0))
    ax.text(0.0, 1.02, "classification accuracy", transform=ax.transAxes,
            fontsize=10, color=INK, ha="left", va="bottom")
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xlim(gx[0] - 0.55, gx[-1] + 0.55)
    ax.set_title("Attack classification accuracy", pad=26, fontsize=12.5)
    ax.grid(axis="y", alpha=0.55); ax.set_axisbelow(True)
    ax.legend(loc="upper left", fontsize=9.5, ncol=2)

    plt.tight_layout(); save_pitch(fig, "12_supervised_accuracy"); plt.show()


### 8.10 Export tables for the write-up

In [ ]:
outdir = Path(INSPECT_PATH).parent
D.to_csv(outdir / "metrics_flat.csv", index=False)

pc_rows = []
for r in rows:
    for head in ("quantum", "dials"):
        for cls, m in r["heads"][head]["per_class"].items():
            pc_rows.append({"config": r["config"], "scheme": r["scheme"], "head": head,
                            "readout_dim": r["readout_dim"], "class": cls, **m})
pd.DataFrame(pc_rows).to_csv(outdir / "metrics_per_class.csv", index=False)
if "OP" in dir() and len(OP):          # only if cell 8.6 has been run
    OP.to_csv(outdir / "metrics_operational.csv", index=False)

print("written:")
for f in sorted(outdir.glob("*.csv")):
    print(f"  {f}  ({f.stat().st_size/1024:.1f} KB)")
print(f"  {outdir/'summary.txt'}")
print(f"  {outdir/'run.log'}")